In [1]:
import os
import requests
import zipfile
import re
import rasterio
import numpy as np
import glob
from rasterio.mask import mask
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.ops import unary_union
import seaborn as sns
import pandas as pd
from scipy import stats
from scipy.stats import skew, kurtosis
from shapely.geometry import Point
from scipy.stats import mode
from shapely.geometry import Point

# Generation des points(long, lat) d'Algerie + Tunisie a partir du shapefile avec une resolution de 1km (prendre un point de la carte chaque 1km)

In [2]:
def download_naturalearth(output_folder="data"):

    """Download Natural Earth shapefile if not present"""
    os.makedirs(output_folder, exist_ok=True)
    shapefile_path = os.path.join(output_folder, "ne_110m_admin_0_countries.shp")

    if not os.path.exists(shapefile_path):
        print("Downloading Natural Earth shapefile...")

        url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
        
        zip_path = os.path.join(output_folder, "ne_countries.zip")

        r = requests.get(url)
        with open(zip_path, "wb") as f:
            f.write(r.content)

        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(output_folder)
        os.remove(zip_path)
        print(" Natural Earth shapefile downloaded and extracted.")
    else:
        print(" Natural Earth shapefile already available.")
    return shapefile_path

In [3]:
 
def get_country_shapes(shapefile_path):
    
    """Extract Algeria and Tunisia polygons"""

    world = gpd.read_file(shapefile_path)

    print(world.columns)
    
    world["COUNTRY"] = world["COUNTRY"].str.strip().str.lower()
    algeria = world[world["COUNTRY"].isin(["algeria", "tunisia"])]

    
    if algeria.empty:
        print(" Could not find Algeria or Tunisia. Available names include:")
        print(world["ADMIN"].unique())
        raise ValueError("Country names not found in shapefile.")

    print(" Found Algeria and Tunisia in shapefile.")
    return algeria



In [4]:

def generate_latlon_grid_csv_fast(country_gdf, resolution_km=1, output_csv="grid_points.csv"):
     
    print("Generating grid points (optimized)...")
 
    metric_crs = "EPSG:32631"
    countries_proj = country_gdf.to_crs(metric_crs)
 
    minx, miny, maxx, maxy = countries_proj.total_bounds
    spacing = resolution_km * 1000  # meters
 
    xs = np.arange(minx, maxx + spacing, spacing)
    ys = np.arange(miny, maxy + spacing, spacing)
    xx, yy = np.meshgrid(xs, ys)
    coords = np.column_stack([xx.ravel(), yy.ravel()])

    print(f"Created {len(coords):,} total grid points before filtering.")
 
    grid_gdf = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(coords[:, 0], coords[:, 1]),
        crs=metric_crs
    )
 
    print("Filtering points inside polygons (using spatial join)...")
    country_union = countries_proj.dissolve().geometry.iloc[0]
    grid_gdf = grid_gdf[grid_gdf.within(country_union)]

   
    grid_gdf = grid_gdf.to_crs("EPSG:4326") 

    
    grid_gdf["longitude"] = grid_gdf.geometry.x
    grid_gdf["latitude"] = grid_gdf.geometry.y

    print(grid_gdf.shape)
    grid_gdf =  grid_gdf[grid_gdf['latitude'] >= 34]
    print(grid_gdf.shape)
    
    
    grid_gdf[["longitude", "latitude"]].to_csv(output_csv, index=False)

    
    print(f"{len(grid_gdf):,} points inside area (resolution: {resolution_km} km).")
    print(f"Saved grid to: {output_csv}")

    return grid_gdf


 


In [5]:
# shapefile_path = r"C:\\Users\melom\OneDrive\Desktop\Data mining\shp\alg_tun.shp"
shapefile_path = download_naturalearth()
country_gdf = get_country_shapes(r"C:\\Users\melom\OneDrive\Desktop\Data mining\shp\alg_tun.shp")

grid_km = generate_latlon_grid_csv_fast(country_gdf, resolution_km=2, output_csv="Cleaned_dataset/grid/algeria_tunisia_grid.csv")

print(grid_km.head())
print(grid_km.shape)




 Natural Earth shapefile already available.
Index(['GID_0', 'COUNTRY', 'geometry'], dtype='object')
 Found Algeria and Tunisia in shapefile.
Generating grid points (optimized)...
Created 1,082,640 total grid points before filtering.
Filtering points inside polygons (using spatial join)...
(618233, 3)
(83312, 3)
83,312 points inside area (resolution: 2 km).
Saved grid to: Cleaned_dataset/grid/algeria_tunisia_grid.csv
                        geometry  longitude   latitude
867695  POINT (2.17897 34.00001)   2.178965  34.000008
867696  POINT (2.20062 34.00015)   2.200620  34.000150
867697  POINT (2.22228 34.00029)   2.222276  34.000289
867698  POINT (2.24393 34.00042)   2.243931  34.000424
867699  POINT (2.26559 34.00056)   2.265586  34.000555
(83312, 3)
